2025-06-25

- saved_eod_2026-06-25.tar.gz, 90MB, on Drive):
  - stage 2:mixmeta(cbind(theta1..5)~1, REML), converged, PSi pos-def and cholesky ok
  pooled 5-vec [0.945,-1.986,-0.081,-4.688,7.110]. first non-deg 3-CMA pool.
  - 3 reduced curves: reduced_tor/reduced_mtl/reduced_van (5-vec + 5x5 each, cen 19.4/19.0/17.0).
  reduced_df (3x7) + vcov_list (3x 5x5) = the Stage 2 frame.
- 3 Stage 1 fits: red_tor/red_mtl_fit/red_van_fit (25-dim coef+vcov+cb_template each).
- tarball also carries 06-17 stragglers (fn_fit_stage1/qaic/reduce_fit, mtl/van substrate,
  cma_age_data_mtlvan) - swept in by save_to_drive
  - didn't bank (seed-42 regenerable): sim, van_sim, mtl_sim, all temp_mats

arc

restore -> C3 (0.726 pass) -> toronto temp_mat+sim rebuilt -> VAN sim built
-> 3 silvers fit (TOR/MTL/VAN 75-84, all variant A and converged) -> 3 reduces
-> stacked -> pool. protected item landed w/ buffer.

what I learned
- death totals scale with pop not Da count. predicted VAN ~78k off DA ratio; got 55,265 (VAN 15.5 deaths/Da vs MTL 22.0 + maritime milder MMT). scale off sum(pop)
- ref_temp is the MEDIAN (50th), not the 80th-pct MMT. predicted MTL/VAN off the MMT band twice, both sat below as they should. don't conflat the two quantiles.
- maritime signature shows at Stage 1: VAN qAIC A lower / B higher than continental, ref_temp 17.0 vs 19.4/19.0 2.4C gap pre-loads C2.
- variant A winning is silver-specific (sparse -> B's weekday strata starve). Not a general result; keep both-variants-pick-by-qAIC. full-scale w/real deaths may flip.
- SOT functions not banked as fn_*.rds must be retyped each restore (hit save_to_drive AND build_crossbasis this session). Bank them as fn_*.rds

Corrections to SSOT
- 06-24 objects not tarball.
- MTL was never actually blocked. dumb.

pick up from: stage 2 exists for 75-85 only. the only open loop is whether to widen (other age bands -> more reduced curves -> ~age_band STage 2) or go down to stage 3 (downscale TOR+VAN, C1 within-CMA + C2 cross-CMA).

restore, clobber order

3 tarballs -> /content. pilot load() first, then 06-17 readRDS clobbwers w/ fixed fns, then 06-24 banked mtl_sim. lib stack reattached by hand.

retyped: build_crossbasis, make_strata_A/B, fit_da_pca - not in any tarball, fit_stage1 calls strata_A/B by name

In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"

system(sprintf("cd /content && cp %s/r_library.tar.gz . && tar -xzf r_library.tar.gz", DRIVE))
.libPaths(c("/content/site-library", .libPaths()))
suppressPackageStartupMessages({
  library(dlnm); library(gnm); library(mixmeta); library(splines)
  library(sf); library(data.table); library(exactextractr); library(terra)
  library(ggplot2); library(viridis); library(lubridate); library(MASS)
})

system(sprintf("cd /content && cp %s/saves_pilot_2026-06-03.tar.gz . && tar -xzf saves_pilot_2026-06-03.tar.gz", DRIVE))
load("/content/saves/pilot_session.RData")

system(sprintf("cd /content && cp %s/saves_eod_2026-06-17.tar.gz . && tar -xzf saves_eod_2026-06-17.tar.gz", DRIVE))
EOD <- "/content/saves_eod"
mtl                 <- readRDS(file.path(EOD, "mtl_substrate.rds"))
van                 <- readRDS(file.path(EOD, "van_substrate.rds"))
cma_age_data_mtlvan <- readRDS(file.path(EOD, "cma_age_data_mtlvan.rds"))
fit_stage1          <- readRDS(file.path(EOD, "fn_fit_stage1.rds"))
qaic                <- readRDS(file.path(EOD, "fn_qaic.rds"))
reduce_fit          <- readRDS(file.path(EOD, "fn_reduce_fit.rds"))

system(sprintf("cd /content && cp %s/saves_eod_2026-06-24.tar.gz . && tar -xzf saves_eod_2026-06-24.tar.gz", DRIVE))
mtl_sim     <- readRDS(file.path(EOD, "mtl_sim.rds"))
mtl_da_long <- readRDS(file.path(EOD, "mtl_da_long.rds"))
mmt_mtl     <- readRDS(file.path(EOD, "mmt_da.rds"))

build_crossbasis <- function(T_series, lag_max = 21) {
  T_knots <- quantile(T_series, probs = c(0.10, 0.75, 0.90), na.rm = TRUE)
  cb <- crossbasis(
    x = T_series, lag = lag_max,
    argvar = list(fun = "bs", degree = 2, knots = T_knots),
    arglag = list(fun = "ns", knots = logknots(lag_max, nk = 3))
  )
  stopifnot(attr(cb, "argvar")$fun == "bs")
  stopifnot(attr(cb, "argvar")$degree == 2)
  cb
}
make_strata_A <- function(DA_id, date) factor(paste(DA_id, year(date), month(date), sep = "_"))
make_strata_B <- function(DA_id, date) factor(paste(DA_id, year(date), month(date), wday(date), sep = "_"))

fit_da_pca <- function(Z_matrix, da_ids, verbose = TRUE) {
  pca <- prcomp(Z_matrix, scale. = TRUE)
  cum_var <- summary(pca)$importance["Cumulative Proportion", 3]
  da_scores <- data.table(DAUID = da_ids, PC1 = pca$x[,1], PC2 = pca$x[,2], PC3 = pca$x[,3])
  if (verbose) cat(sprintf("PCA: cum var first 3 = %.1f%%\n", 100*cum_var))
  stopifnot(cum_var > 0.5)
  list(pca = pca, scores = da_scores, var_explained = summary(pca)$importance["Proportion of Variance", 1:3])
}

cat("--- library ---\n")
cat("fread:", exists("fread"), " crossbasis:", exists("crossbasis"), "\n")
cat("--- MTL sim ---\n")
cat("rows:", nrow(mtl_sim), " deaths:", sum(mtl_sim$n_deaths), "\n")
cat("--- substrates ---\n")
cat("mtl:", paste(dim(mtl$temp_mat), collapse="x"), " van:", paste(dim(van$temp_mat), collapse="x"), "\n")
cat("--- fns ---\n")
cat("fit_stage1:", exists("fit_stage1"), " qaic:", exists("qaic"), " reduce_fit:", exists("reduce_fit"),
    " build_crossbasis:", exists("build_crossbasis"), " strata_A/B:", exists("make_strata_A"), exists("make_strata_B"), "\n")
cat("objects:", length(ls()), "\n")

Warning message in gzfile(file, "rb"):
“cannot open compressed file '/content/saves_eod/mtl_sim.rds', probable reason 'No such file or directory'”


ERROR: Error in gzfile(file, "rb"): cannot open the connection


In [ ]:
system(sprintf("cp %s/mtl_sim_2026-06-24.rds /content/", DRIVE))
system(sprintf("cp %s/mtl_da_long_2026-06-24.rds /content/", DRIVE))
system(sprintf("cp %s/mmt_da_mtl_2026-06-24.rds /content/", DRIVE))

mtl_sim     <- readRDS("/content/mtl_sim_2026-06-24.rds")
mtl_da_long <- readRDS("/content/mtl_da_long_2026-06-24.rds")
mmt_mtl     <- readRDS("/content/mmt_da_mtl_2026-06-24.rds")

cat("mtl_sim     rows:", nrow(mtl_sim), " deaths:", sum(mtl_sim$n_deaths), "\n")
cat("mtl_da_long rows:", nrow(mtl_da_long), " NA pop:", sum(is.na(mtl_da_long$pop)), "\n")
cat("mmt_mtl     length:", length(mmt_mtl), " range:", paste(round(range(mmt_mtl),1), collapse="-"), "\n")
cat("mtl$n_da:", mtl$n_da, " — mmt matches n_da:", length(mmt_mtl) == mtl$n_da, "\n")

mtl_sim     rows: 19902240  deaths: 142934 
mtl_da_long rows: 19902240  NA pop: 0 
mmt_mtl     length: 6504  range: 20.5-22.5 
mtl$n_da: 6504  — mmt matches n_da: TRUE 


mtl_sim     rows: 19902240  deaths: 142934
mtl_da_long rows: 19902240  NA pop: 0
mmt_mtl     length: 6504  range: 20.5-22.5
mtl$n_da: 6504  — mmt matches n_da: TRUE

---

the 06-24 MTl are objects not a tarball, had it wrong. on my drive there are three loose dated files

mtl_sim_2026-06-24.rds / mtl_da_long_2026-06-24.rds / mmt_da_mtl_2026-06-24.rds. No such tarball exist. Fixing SSOT

---

restore complete, MTl banked sim verified.

sim + da_long agree at ~19M rows, 0 NA pop -> 142,934 deaths, exact match. mmt_mtl 6504 long, range 20.5--2.5 one 80th pct MMT per Da, tight band (MTL warm-season is uniform across DAs, make sense as it is a small geo spread). nothing surprising

C3 PCA truth recovery (independent, no fits needed)

prcomp(scale=TRUE) on Toronto Z (17 vuln varbs, latents baked in via L).
correlate empirical loadings vs true_loadings L. match across 6 permutations, allow sign flip, pass = mean |cor| > 0.7. expect ~0.73: F1, F2 clean ~0.85 each, F3 ~0.49 (entangled w/ F1 via L's cross loadings rows 16-17.

In [ ]:
pca_out  <- fit_da_pca(Z, truth_factors$DAUID)
emp_load <- pca_out$pca$rotation[, 1:3]
cormat   <- cor(emp_load, true_loadings)

perms    <- rbind(c(1,2,3), c(1,3,2), c(2,1,3), c(2,3,1), c(3,1,2), c(3,2,1))
best_p   <- perms[which.max(apply(perms, 1, function(p) mean(abs(diag(cormat[p, ]))))), ]
diag_cor <- abs(diag(cormat[best_p, ]))

cat("matched |cor| per factor:", round(diag_cor, 3), "\n")
cat("mean |cor|:", round(mean(diag_cor), 3), "  threshold 0.70\n")
cat("per-true-F max |cor|:", round(apply(abs(cormat), 2, max), 3), "\n")

  PCA: var explained PC1-3 = 31.4%, 27.6%, 25.5% (cum 84.5%)
matched |cor| per factor: 0.848 0.851 0.48 
mean |cor|: 0.726   threshold 0.70
per-true-F max |cor|: 0.848 0.851 0.751 


 PCA: var explained PC1-3 = 31.4%, 27.6%, 25.5% (cum 84.5%)
matched |cor| per factor: 0.848 0.851 0.48
mean |cor|: 0.726   threshold 0.70
per-true-F max |cor|: 0.848 0.851 0.751

---

C# pass, mean |cor| 0.726 > 0.70.

F1, F2 recovered clean: 0.848, 0.851 -> two latent factors map 1:1 to distinct PCs.



read toronto_daymet csv, pivot DA x day, align row order to truth_factors via match()

I am predicting 12 water-only DAs because 9180 NA/765 = 12.

In [ ]:
da_daily_temp <- fread(daymet_csv, select = c("DAUID","date","tmax","tmin"))
da_daily_temp[, DAUID := as.character(DAUID)]
da_daily_temp[, date  := as.Date(date)]
da_daily_temp[, tmean_C := (tmax + tmin) / 2]

temp_wide <- dcast(da_daily_temp, DAUID ~ date, value.var = "tmean_C")
temp_wide <- temp_wide[match(truth_factors$DAUID, temp_wide$DAUID), ]
temp_mat  <- as.matrix(temp_wide[, -1])
rownames(temp_mat) <- temp_wide$DAUID

dead_rows <- which(rowSums(is.na(temp_mat)) == ncol(temp_mat))
keep      <- setdiff(seq_len(nrow(temp_mat)), dead_rows)

temp_mat      <- temp_mat[keep, ]
truth_factors <- truth_factors[keep, ]
Z             <- Z[keep, ]
n_da          <- length(keep)
truth_factors[, da_idx := .I]

cat("Dropped water-only DAs:", length(dead_rows), "\n")
cat("temp_mat:", paste(dim(temp_mat), collapse=" x "),
    " alignment:", all(rownames(temp_mat) == truth_factors$DAUID),
    " NA:", sum(is.na(temp_mat)), "\n")
cat("range:", paste(round(range(temp_mat, na.rm=TRUE),1), collapse="-"), "  n_da:", n_da, "\n")

Dropped water-only DAs: 12 
temp_mat: 7682 x 765  alignment: TRUE  NA: 0 
range: 2.2-30.1   n_da: 7682 


Dropped water-only DAs: 12
temp_mat: 7682 x 765  alignment: TRUE  NA: 0
range: 2.2-30.1   n_da: 7682

---

12 water DAs dropped exactly. alignment TRUE -> temp_mat rows pair 1:1 to truth_Factors, deaths will attach to correct weather. NA 0, range 2.2-30.1C sane warm-season. This is toronto bec. 7682 x 765. Feeds the sim

now doing toronto sim part 2. Rebuilding da_long on trimmed 7682 set so da_idx matches temp_mat rows, melt pops in, compute lambda0.

Predictions: da_long rows = 23,506,920 (7682 x 765 x 4). NA pop 0 (every Da-band has a population). Total deaths ~ 189,137 to match section 5.5, seed 42 is det. Age gradient must be climbing from 10k to 43k to 58k to 77k. Oldest band fewest people most deaths

In [ ]:
da_age_trim <- da_age[match(truth_factors$DAUID, da_age$ALT_GEO_CODE)]
da_age_trim[, da_idx := .I]

da_long <- CJ(da_idx = 1:n_da, date = study_dates, age_band = names(annual_rates))
pop_long <- melt(da_age_trim[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
                 id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
pop_long[, age_band := as.character(age_band)]
da_long <- pop_long[da_long, on = c("da_idx","age_band")]
da_long[, annual_rate := annual_rates[age_band]]
da_long[, lambda0 := pop * annual_rate / 1000 / 365]

mmt_da <- apply(temp_mat, 1, function(x) quantile(x, 0.80, na.rm=TRUE))
sim <- simulate_counts(temp_mat, da_long, truth_factors, mmt_da, seed = 42)

cat("da_long rows:", nrow(da_long), " NA pop:", sum(is.na(da_long$pop)), "\n")
cat("total deaths:", sum(sim$n_deaths), " NA:", sum(is.na(sim$n_deaths)), "\n")
print(sim[, .(deaths = sum(n_deaths)), by = age_band])

Simulating 7682 DAs × 765 days × 4 age bands
Total deaths simulated: 189137 
Mean deaths per DA-day-age: 0.008 
% zero days: 99.2 %
da_long rows: 23506920  NA pop: 0 
total deaths: 189137  NA: 0 
    age_band deaths
      <char>  <int>
1:  age_0_64  10233
2: age_65_74  43381
3: age_75_84  58313
4:   age_85p  77210


Simulating 7682 DAs × 765 days × 4 age bands
Total deaths simulated: 189137
Mean deaths per DA-day-age: 0.008
% zero days: 99.2 %
da_long rows: 23506920  NA pop: 0
total deaths: 189137  NA: 0
    age_band deaths
      <char>  <int>
1:  age_0_64  10233
2: age_65_74  43381
3: age_75_84  58313
4:   age_85p  77210

---

Predictions check out exactly. MTL substrate + sim + mmt_mtl restored, Toronto: substrate + sim + mmt_da regenerated, VAN substrate only sim not yet built, coming next

Vancouver sim. MTL and Toronto sims are both done, VAN has its substrate (van$temp_mat, van$truth_Factors, van$Z, van$da_age, 3575 DAs) no mortality, but same DGP as the other two; just swapping mtl -> van

predictions: van_da_long rows = 10,933,380 since 3575 x 765 x 4. VAN is smallest city, half of MTl. NA pop = 0. No total deaths banked # to check against since VAN sim has never run. But we expect ~78,000 deaths since MTL 6504 DAs -> 142,934 deaths, by order of mag VAN 3573 DAs = roughly half of MTL. Age gradient still climbing (same rate structure as MTL)

as extra. check, I wrote mmt_Van range. VAN is maritime, its 80th-pct temps should sit lower than MTL's 20.5-22.5, so expecting 17-20.

In [ ]:
mmt_van <- apply(van$temp_mat, 1, function(x) quantile(x, 0.80, na.rm = TRUE))

van_da_long <- CJ(da_idx = 1:van$n_da, date = study_dates, age_band = names(annual_rates))
pop_long <- melt(van$da_age[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
                 id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
pop_long[, age_band := as.character(age_band)]
van_da_long <- pop_long[van_da_long, on = c("da_idx", "age_band")]
van_da_long[, annual_rate := annual_rates[age_band]]
van_da_long[, lambda0 := pop * annual_rate / 1000 / 365]

van_sim <- simulate_counts(van$temp_mat, van_da_long, van$truth_factors, mmt_van, seed = 42)

cat("van_da_long rows:", nrow(van_da_long), " NA pop:", sum(is.na(van_da_long$pop)), "\n")
cat("total deaths:", sum(van_sim$n_deaths), " NA:", sum(is.na(van_sim$n_deaths)), "\n")
cat("mmt_van range:", paste(round(range(mmt_van),1), collapse="-"), "\n")
print(van_sim[, .(deaths = sum(n_deaths)), by = age_band])

Simulating 3573 DAs × 765 days × 4 age bands
Total deaths simulated: 55265 
Mean deaths per DA-day-age: 0.0051 
% zero days: 99.5 %
van_da_long rows: 10933380  NA pop: 0 
total deaths: 55265  NA: 0 
mmt_van range: 17.4-20.7 
    age_band deaths
      <char>  <int>
1:  age_0_64   2748
2: age_65_74  13412
3: age_75_84  16951
4:   age_85p  22154


Simulating 3573 DAs × 765 days × 4 age bands
Total deaths simulated: 55265
Mean deaths per DA-day-age: 0.0051
% zero days: 99.5 %
van_da_long rows: 10933380  NA pop: 0
total deaths: 55265  NA: 0
mmt_van range: 17.4-20.7
    age_band deaths
      <char>  <int>
1:  age_0_64   2748
2: age_65_74  13412
3: age_75_84  16951
4:   age_85p  22154

 ---

VAN Sim built, det, deaths lower than predicted (Explained above), rows 10,933,380 exaxct, NA 0, age gradient climbs 2748 -> 22154. lower temp for mmt_van -> maritime -> sane.

now the fit on toronto 75-84. pull 150 random DAs with seed 42, subset sim to those DAs + age 75-84, attach each DA's temp in matching row order, build cb, fit_stage1 both variants, then crossreduce at TOronto's median temp.

predictions: cross-basis dims 114750 x 25 since 150 DAs. 765 = 114,750 rows; 25 = 5 temp x 5 lag. variant A must win since seed 42 + same silver. coef length 25 vcov 25 x 25, no NA, and it must be underdispersed since this is clean simulated Poisson

In [ ]:
set.seed(42)
sliver_idx <- sample(unique(sim$da_idx), 150)

sliver <- sim[da_idx %in% sliver_idx & age_band == "age_75_84"]
sliver[, DA_id := da_idx]
setorder(sliver, da_idx, date)

temp_long <- data.table(
  da_idx = rep(sliver_idx, each = 765),
  date   = rep(study_dates, times = 150),
  temp_C = as.vector(t(temp_mat[sliver_idx, ]))
)
sliver <- temp_long[sliver, on = c("da_idx","date")]

cb  <- build_crossbasis(sliver$temp_C, lag_max = 21)
res <- fit_stage1(sliver, cb, "Toronto", "age_75_84")

cat("Cross-basis:", paste(dim(cb), collapse=" x "), "\n")
cat("Winner:", res$winner_variant,
    " qAIC A:", round(res$qaic_A,1), " qAIC B:", round(res$qaic_B,1), "\n")
cat("coef:", length(res$coef), " vcov:", paste(dim(res$vcov), collapse=" x "),
    " any NA:", any(is.na(res$coef)), "\n")


=== Stage 1: Toronto, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: qAIC = 28979.4
  Variant B: 26250 strata, 114750 rows
  Variant B: qAIC = 72114.6
  Winner: variant A
Cross-basis: 114750 x 25 
Winner: A  qAIC A: 28979.4  qAIC B: 72114.6 
coef: 25  vcov: 25 x 25  any NA: FALSE 


=== Stage 1: Toronto, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: qAIC = 28979.4
  Variant B: 26250 strata, 114750 rows
  Variant B: qAIC = 72114.6
  Winner: variant A
Cross-basis: 114750 x 25
Winner: A  qAIC A: 28979.4  qAIC B: 72114.6
coef: 25  vcov: 25 x 25  any NA: FALSE

---

qAIC matches bank to the decimal. A wins 28,979.4 vs B 72,114.6. fit machinery reproduces known-good in this session. Strata explains why: A 3,750 vs B 26,250 (7x finer). B's near-empty weekday strata -> no power -> qAIC crushes it. CB 114,750x25, coef 25, vcov 25x25 no NA.

reduce toronto 75-84 by colapsing the 25-dim cb to its 5-dim overall cum curve as per Gasp-arm 2013, reduce-before-pool. 25-dim phi has 325 free elements, unestimable; reduced 5-dim phi has 15, poolable. crossreduce type=overall, centered at Toronto median warm-season temp.

predictions:
- ref_temp ~ 19.4C
- theta_Star length = 5, V_star = 5x5 no Na
- this is first of 3 reduced curves stage 2 frame needs

In [ ]:
ref_temp <- median(sliver$temp_C)
red <- reduce_fit(res, ref_temp)

cat("ref_temp:", round(ref_temp,1), "\n")
cat("theta_star:", length(red$theta_star),
    " V_star:", paste(dim(red$V_star), collapse=" x "),
    " any NA:", any(is.na(red$theta_star)), "\n")

ref_temp: 19.4 
theta_star: 5  V_star: 5 x 5  any NA: FALSE 


ref_temp: 19.4
theta_star: 5  V_star: 5 x 5  any NA: FALSE

---

ref_temp 19.4 = banked median so silver reproduced. theta_star 5-vec = lag dim collapsed, overall cum log-RR vs temp. V_star 5x5, no NA.

### State so far

1 of 3 reduced curves banked
- Toronto 75-84: reduced
- MTL 75-84: fit + reduce next
- VAN 75-84: fit + reduce after.
then stack 3 -> stage 2

MTL 75-84 silver - first real MTL fit. 150 MTL DAs seed 42, age 75-84, attach temp from mtl$temp_mat by da_idx, build cb, fit_stage1 both variants, qAIC picks

pred:
- cb 114750 x 25 since 150 DA x 765 = 114750 rows; 25 = 5 temp x 5 lag, same basis spec as TOronto
- Variant A wins again since silver is equally sparse for MTL.
- qAIC magnitude near TOronto's order since similar row couunt and death density (MTL 75-84 had 45,847 full-city deaths vs Toronto 58,311, same ballpark.

In [ ]:
set.seed(42)
mtl_sliver_idx <- sample(unique(mtl_sim$da_idx), 150)

mtl_sliver <- mtl_sim[da_idx %in% mtl_sliver_idx & age_band == "age_75_84"]
mtl_sliver[, DA_id := da_idx]
setorder(mtl_sliver, da_idx, date)

mtl_temp_long <- data.table(
  da_idx = rep(mtl_sliver_idx, each = 765),
  date   = rep(study_dates, times = 150),
  temp_C = as.vector(t(mtl$temp_mat[mtl_sliver_idx, ]))
)
mtl_sliver <- mtl_temp_long[mtl_sliver, on = c("da_idx","date")]

cb_mtl  <- build_crossbasis(mtl_sliver$temp_C, lag_max = 21)
res_mtl <- fit_stage1(mtl_sliver, cb_mtl, "Montreal", "age_75_84")

cat("Cross-basis:", paste(dim(cb_mtl), collapse=" x "), "\n")
cat("Winner:", res_mtl$winner_variant,
    " qAIC A:", round(res_mtl$qaic_A,1), " qAIC B:", round(res_mtl$qaic_B,1), "\n")
cat("coef:", length(res_mtl$coef), " vcov:", paste(dim(res_mtl$vcov), collapse=" x "),
    " any NA:", any(is.na(res_mtl$coef)), "\n")


=== Stage 1: Montreal, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: qAIC = 28999.8
  Variant B: 26250 strata, 114750 rows
  Variant B: qAIC = 72134.8
  Winner: variant A
Cross-basis: 114750 x 25 
Winner: A  qAIC A: 28999.8  qAIC B: 72134.8 
coef: 25  vcov: 25 x 25  any NA: FALSE 


=== Stage 1: Montreal, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: qAIC = 28999.8
  Variant B: 26250 strata, 114750 rows
  Variant B: qAIC = 72134.8
  Winner: variant A
Cross-basis: 114750 x 25
Winner: A  qAIC A: 28999.8  qAIC B: 72134.8
coef: 25  vcov: 25 x 25  any NA: FALSE

---

A wins again - within 20 pts or Toronto's 28,979/72,114. two cities but same qAIC regime -> machinery + silver sparsity behave identically.

reduce MTL 75-84: same 25->5 collapse, MTL's fit, cen = MTL median temp (its own, not toronto's 19.4)

pred:
- ref_temp_mtl ~ 20-21: median of MTl silver temp; Mtl warm-season runs slightly warmer/tighter than Tor (recall mmt_mtl band 20.5 - 22.5), so median sits a touch above 19.4, not 19.4 - diff city
- theta_star 5, V_star 5x5, no NA since crossreduce overall collapse lag dim same as Tor reduce

In [ ]:
ref_temp_mtl <- median(mtl_sliver$temp_C)
red_mtl <- reduce_fit(res_mtl, ref_temp_mtl)

cat("ref_temp_mtl:", round(ref_temp_mtl,1), "\n")
cat("theta_star:", length(red_mtl$theta_star),
    " V_star:", paste(dim(red_mtl$V_star), collapse=" x "),
    " any NA:", any(is.na(red_mtl$theta_star)), "\n")

ref_temp_mtl: 19 
theta_star: 5  V_star: 5 x 5  any NA: FALSE 


ref_temp_mtl: 19
theta_star: 5  V_star: 5 x 5  any NA: FALSE

---

19.0 flagged as low vs predicted 20-21, but I predicted off 80th-pct MMT band, and ref_temp is the median, which sits below. 19.0 is consistent, distinct from 19.4 of Toronto -> not identical -> cities didn't cross

van 75-84 silver - last fit before stage 2

150 VAn DAs seed 42, age 75-84, temp from van$temp_mat by da_idx, build cb, fit_stage1 both variants, qAIC pick
VAN maritime -> built-in check its curve should differ from the 2 cont. (tor and mtl)

In [ ]:
set.seed(42)
van_sliver_idx <- sample(unique(van_sim$da_idx), 150)

van_sliver <- van_sim[da_idx %in% van_sliver_idx & age_band == "age_75_84"]
van_sliver[, DA_id := da_idx]
setorder(van_sliver, da_idx, date)

van_temp_long <- data.table(
  da_idx = rep(van_sliver_idx, each = 765),
  date   = rep(study_dates, times = 150),
  temp_C = as.vector(t(van$temp_mat[van_sliver_idx, ]))
)
van_sliver <- van_temp_long[van_sliver, on = c("da_idx","date")]

cb_van  <- build_crossbasis(van_sliver$temp_C, lag_max = 21)
res_van <- fit_stage1(van_sliver, cb_van, "Vancouver", "age_75_84")

cat("Cross-basis:", paste(dim(cb_van), collapse=" x "), "\n")
cat("Winner:", res_van$winner_variant,
    " qAIC A:", round(res_van$qaic_A,1), " qAIC B:", round(res_van$qaic_B,1), "\n")
cat("coef:", length(res_van$coef), " vcov:", paste(dim(res_van$vcov), collapse=" x "),
    " any NA:", any(is.na(res_van$coef)), "\n")


=== Stage 1: Vancouver, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: qAIC = 27253.1
  Variant B: 26250 strata, 114750 rows
  Variant B: qAIC = 75012.1
  Winner: variant A
Cross-basis: 114750 x 25 
Winner: A  qAIC A: 27253.1  qAIC B: 75012.1 
coef: 25  vcov: 25 x 25  any NA: FALSE 



=== Stage 1: Vancouver, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: qAIC = 27253.1
  Variant B: 26250 strata, 114750 rows
  Variant B: qAIC = 75012.1
  Winner: variant A
Cross-basis: 114750 x 25
Winner: A  qAIC A: 27253.1  qAIC B: 75012.1
coef: 25  vcov: 25 x 25  any NA: FALSE

---

A wins 27,253.1 vs B 75,012.1 distinct from cotinental (TOr 28,979/72,114, MTL 28,999/72,134): VAN A's lower, B is higher! maritime -> thinner heat signal -> A fits sparser response cheaper, B over-stratifies worse. again not identical to MTL = keying held, cities didn't cross.

reduce VAN 75-84: 25 -> 5 collapse, VAN's fit, cen = VAN median temp.
maritime -> ref_temp should sit below toronto 19.4 and MTL 19.0

pred: ref_temp_Van ~ 16-18: why because median of VAn silver temp; maritime climate runs milder (recall mmt_van 80th-pct 17.4-20.7) so the median sits below that band, lower than both continential medians. distinctly <19
- theta_star 5, V_star 5x5, no NA - same crossreduce overall collapse

In [ ]:
ref_temp_van <- median(van_sliver$temp_C)
red_van <- reduce_fit(res_van, ref_temp_van)

cat("ref_temp_van:", round(ref_temp_van,1), "\n")
cat("theta_star:", length(red_van$theta_star),
    " V_star:", paste(dim(red_van$V_star), collapse=" x "),
    " any NA:", any(is.na(red_van$theta_star)), "\n")

ref_temp_van: 17 
theta_star: 5  V_star: 5 x 5  any NA: FALSE 


ref_temp_van: 17
theta_star: 5  V_star: 5 x 5  any NA: FALSE

---

17.0 distinctly below tor 19.4 + MTl 19.0 - 2.4C gap. maritime signature in ref temp itself. cross-CMA contrast pre-loading (C2 wants >= 3C MMT gap; already 2.4 in ref_temp before MMT computed). STAGE 2 COMPLETE!

### STate: 3 of 3 reduced curves now done

- Toronto 75-84: reduced (cen 19.4)
- MTL 75-84: reduced (cen 19.0)
- VAN 75-84: reduced (cen 17.0)

assemble stage 2 frame by stacking the 3 reduced curves into mixmeta shape. reduced_df = CMA, age_band, theta1..5 (3 rows). vcov_list = 3 matching 5x5. 1 band not 4 -> frame is 3 rows (3 CMA x 1 band), not 12. Stage 2 will be intercept-only (~1), still fthe first non-degenerate 3-CMA pool

pred:
- reduced_df 3 rows x (2 label + 5 theta) cols: 3 cities, 1 band each
- vcov_list length 3, each 5x5: one V_star per reduced curve
- no NA in theta or vcov: all 3 reduces came back clean

In [ ]:
reduced_list <- list(
  list(cma = "Toronto",   age = "age_75_84", theta = red$theta_star,     V = red$V_star),
  list(cma = "Montreal",  age = "age_75_84", theta = red_mtl$theta_star, V = red_mtl$V_star),
  list(cma = "Vancouver", age = "age_75_84", theta = red_van$theta_star, V = red_van$V_star)
)

theta_mat  <- t(sapply(reduced_list, function(x) x$theta))
colnames(theta_mat) <- paste0("theta", 1:5)
vcov_list  <- lapply(reduced_list, function(x) x$V)

reduced_df <- data.table(
  CMA      = sapply(reduced_list, function(x) x$cma),
  age_band = sapply(reduced_list, function(x) x$age)
)
reduced_df <- cbind(reduced_df, as.data.table(theta_mat))

cat("reduced_df:", paste(dim(reduced_df), collapse=" x "), "\n")
print(reduced_df)
cat("vcov_list length:", length(vcov_list),
    " each:", paste(dim(vcov_list[[1]]), collapse="x"), "\n")
cat("theta NA:", sum(is.na(theta_mat)),
    " vcov NA:", sum(sapply(vcov_list, function(v) sum(is.na(v)))), "\n")

reduced_df: 3 x 7 
         CMA  age_band    theta1     theta2    theta3    theta4    theta5
      <char>    <char>     <num>      <num>     <num>     <num>     <num>
1:   Toronto age_75_84  4.568560 -0.4288152  1.469894 -4.495881 20.264623
2:  Montreal age_75_84 -2.985385 -4.5637742 -3.262888 -4.598809 -7.040696
3: Vancouver age_75_84  1.564383 -0.7111874  1.817521 -4.652593  8.155695
vcov_list length: 3  each: 5x5 
theta NA: 0  vcov NA: 0 


reduced_df: 3 x 7
         CMA  age_band    theta1     theta2    theta3    theta4    theta5
      <char>    <char>     <num>      <num>     <num>     <num>     <num>
1:   Toronto age_75_84  4.568560 -0.4288152  1.469894 -4.495881 20.264623
2:  Montreal age_75_84 -2.985385 -4.5637742 -3.262888 -4.598809 -7.040696
3: Vancouver age_75_84  1.564383 -0.7111874  1.817521 -4.652593  8.155695
vcov_list length: 3  each: 5x5
theta NA: 0  vcov NA: 0

---

3x7 (CMA, age_band, theta1-5), NA 0 across theta + vcov. theta rows differ city to city (Tor theta5 20.3, MTl -7.0, VAN 8.2) -> cities provably separate, keying held E2E. MTl all-negative = centering artifact (each curve cen'd at own median 19.4/19.0/17.0, diff baselines, not comparable raw - vcov carries it into the pool).

pool - first non-degenerate 3-CMA stage 2. int-only (~1, one band), inline not SOt fit_Stage2 (which hardcodes PCs + age_band, chokes here). REML first; fixed-eff fallback if phi won't estimate from 3 groups (expected). retires the N_CMA=1 fallback.

pred:

- converges (REML or fixed): 3 obs, int-only, 5-dim outcome; small but non-deg. REML's 5-dim phi from 3 groups may fail -> fixed fallback
- pooled coef = 5-vec: int-only multivariate meta returns one 5-vector (across-city average curve)
- Cholesky passes if REML + phi pos-def; if fixed, no phi to check

In [ ]:
stage2 <- tryCatch(
  mixmeta(cbind(theta1, theta2, theta3, theta4, theta5) ~ 1,
          S = vcov_list, data = reduced_df, method = "reml"),
  error = function(e) { cat("REML failed:", conditionMessage(e), "\n-> fixed fallback\n"); NULL }
)

if (is.null(stage2)) {
  stage2 <- mixmeta(cbind(theta1, theta2, theta3, theta4, theta5) ~ 1,
                    S = vcov_list, data = reduced_df, method = "fixed")
  method_used <- "fixed"
} else {
  method_used <- "reml"
}

cat("method:", method_used, "\n")
cat("converged:", isTRUE(stage2$converged), "\n")
cat("pooled coef length:", length(coef(stage2)), "\n")
print(round(coef(stage2), 3))
if (method_used == "reml") {
  chol_ok <- tryCatch({ chol(stage2$Psi); TRUE }, error = function(e) FALSE)
  cat("Cholesky (Ψ pos-def):", chol_ok, "\n")
}

method: reml 
converged: TRUE 
pooled coef length: 5 
theta1 theta2 theta3 theta4 theta5 
 0.945 -1.986 -0.081 -4.688  7.110 
Cholesky (Ψ pos-def): TRUE 


method: reml
converged: TRUE
pooled coef length: 5
theta1 theta2 theta3 theta4 theta5
 0.945 -1.986 -0.081 -4.688  7.110
Cholesky (Ψ pos-def): TRUE  

---

REML converged method reml (not fixed - better path than predicted). converged true. CHolesky true -> psi (between-city random cov) pos-def, valid.
3 groups carried a 5-dim psi - didn't expect it to hold but it held.
pooled coef = 5-vec [0.945, -1.986, -4.688, 7.110] = across-city avg curve, inverse-vcov weighted.

In [ ]:
session_keepers <- list(
  red_tor     = res,        # Toronto 75-84 Stage 1 fit (coef 25, vcov 25x25, cb_template)
  red_mtl_fit = res_mtl,    # MTL 75-84 Stage 1 fit
  red_van_fit = res_van,    # VAN 75-84 Stage 1 fit
  reduced_tor = red,        # Toronto reduced curve (theta_star 5, V_star 5x5, cen 19.4)
  reduced_mtl = red_mtl,    # MTL reduced (cen 19.0)
  reduced_van = red_van,    # VAN reduced (cen 17.0)
  reduced_df  = reduced_df, # Stage 2 frame, 3x7
  vcov_list   = vcov_list,  # 3 x 5x5
  stage2      = stage2      # THE POOL: mixmeta ~1 reml, converged, Psi pos-def
)

save_to_drive(session_keepers, tag = "2026-06-25")

ERROR: Error in save_to_drive(session_keepers, tag = "2026-06-25"): could not find function "save_to_drive"


In [ ]:
save_to_drive <- function(objs, tag = format(Sys.Date(), "%Y-%m-%d"), drive = DRIVE) {
  dir.create("/content/saves_eod", showWarnings = FALSE)
  for (nm in names(objs)) saveRDS(objs[[nm]], sprintf("/content/saves_eod/%s.rds", nm))
  tarball <- sprintf("saves_eod_%s.tar.gz", tag)
  system(sprintf("cd /content && tar -czf %s saves_eod/ && cp %s '%s/'", tarball, tarball, drive))
  cat("saved", length(objs), "objects ->", file.path(drive, tarball), "\n")
}

session_keepers <- list(
  red_tor     = res,
  red_mtl_fit = res_mtl,
  red_van_fit = res_van,
  reduced_tor = red,
  reduced_mtl = red_mtl,
  reduced_van = red_van,
  reduced_df  = reduced_df,
  vcov_list   = vcov_list,
  stage2      = stage2
)

save_to_drive(session_keepers, tag = "2026-06-25")

saved 9 objects -> /content/drive/MyDrive/thesis/dlnm-pilot/saves_eod_2026-06-25.tar.gz 


In [ ]:
cat(system(sprintf("ls -la %s/saves_eod_2026-06-25.tar.gz 2>&1", DRIVE), intern = TRUE), sep = "\n")
cat("\n--- contents ---\n")
cat(system("tar -tzf /content/saves_eod_2026-06-25.tar.gz 2>&1", intern = TRUE), sep = "\n")

-rw------- 1 root root 89753098 Jun 25 19:37 /content/drive/MyDrive/thesis/dlnm-pilot/saves_eod_2026-06-25.tar.gz

--- contents ---
saves_eod/
saves_eod/fn_fit_stage1.rds
saves_eod/van_substrate.rds
saves_eod/reduced_mtl.rds
saves_eod/stage2.rds
saves_eod/vcov_list.rds
saves_eod/mtl_substrate.rds
saves_eod/cma_age_data_mtlvan.rds
saves_eod/fn_qaic.rds
saves_eod/red_van_fit.rds
saves_eod/red_mtl_fit.rds
saves_eod/red_tor.rds
saves_eod/reduced_tor.rds
saves_eod/fn_reduce_fit.rds
saves_eod/reduced_van.rds
saves_eod/reduced_df.rds
